# 03 EDA and Feature Review

This notebook reviews the engineered customer feature table before clustering. The goal is to understand the feature groups, identify useful modeling candidates, and separate them from fields that are better used only for profiling or validation.

No clustering model is trained in this notebook, and no final clustering output file is created.


## Purpose of this notebook

The data audit and preprocessing notebooks already checked the raw data and built a clean customer-level feature table. This phase asks a different question: which engineered features are useful for clustering, and which ones should only help us explain the clusters afterwards?

The notebook keeps the review simple and defense-friendly. It uses validation checks, compact summary tables, and a few optional plots instead of a large exploratory analysis.


## Setup and data loading

This section locates the project root, imports the existing project helpers, and loads the raw datasets. The fixed reference date makes age and tenure calculations reproducible.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    PLOTS_AVAILABLE = True
except ImportError:
    plt = None
    PLOTS_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)


def has_raw_datasets(candidate):
    root_layout = (candidate / "customer_info.csv").exists() and (candidate / "customer_basket.csv").exists()
    project_files_layout = (candidate / "Project files" / "customer_info.csv").exists() and (candidate / "Project files" / "customer_basket.csv").exists()
    return root_layout or project_files_layout


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if has_raw_datasets(candidate):
            return candidate
    raise FileNotFoundError("Could not locate raw datasets at the repository root or in Project files/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE_DATE = pd.Timestamp("2026-05-30")
print(f"Project root: {PROJECT_ROOT}")
print(f"Reference date for age and tenure: {REFERENCE_DATE.date()}")
print(f"Matplotlib available for optional plots: {PLOTS_AVAILABLE}")


In [ ]:
from src.data_loading import load_datasets
from src.features import build_customer_feature_table


In [ ]:
customer_info, customer_basket = load_datasets(PROJECT_ROOT)

print(f"customer_info rows: {len(customer_info):,}")
print(f"unique customers in customer_info: {customer_info['customer_id'].nunique():,}")
print(f"customer_basket rows: {len(customer_basket):,}")


## Build and validate feature table

The feature table is built with the existing project workflow. The validation checks confirm that the table is still complete, clean, and safe to review before clustering.


In [ ]:
feature_table, metadata = build_customer_feature_table(
    customer_info,
    customer_basket,
    reference_date=REFERENCE_DATE,
)

print(f"feature table shape: {feature_table.shape}")
print(f"output customer count: {metadata['output_customer_count']:,}")
print(f"basket parse errors: {metadata['basket_parse_errors']}")
print(f"customers without sampled baskets: {metadata['customers_without_baskets']:,}")


In [ ]:
degree_columns = ["degree_bsc", "degree_msc", "degree_phd", "degree_unknown"]
outputs_dir = PROJECT_ROOT / "outputs"
final_output_files = []

if outputs_dir.exists():
    for path in outputs_dir.iterdir():
        name = path.name.lower()
        looks_like_final_cluster_output = path.suffix.lower() == ".csv" and (
            "cluster" in name or "segment" in name
        )
        if path.is_file() and looks_like_final_cluster_output:
            final_output_files.append(path.name)

validation_results = pd.DataFrame(
    [
        {"check": "expected customer count", "value": metadata["output_customer_count"], "passes": metadata["output_customer_count"] == 33038},
        {"check": "expected feature table shape", "value": str(feature_table.shape), "passes": feature_table.shape == (33038, 77)},
        {"check": "customer_id is unique", "value": feature_table["customer_id"].is_unique, "passes": feature_table["customer_id"].is_unique},
        {"check": "missing values", "value": int(feature_table.isna().sum().sum()), "passes": int(feature_table.isna().sum().sum()) == 0},
        {"check": "customer_name excluded", "value": "customer_name" in feature_table.columns, "passes": "customer_name" not in feature_table.columns},
        {"check": "degree columns present", "value": all(column in feature_table.columns for column in degree_columns), "passes": all(column in feature_table.columns for column in degree_columns)},
        {"check": "degree flags sum to one", "value": bool((feature_table[degree_columns].sum(axis=1) == 1).all()), "passes": bool((feature_table[degree_columns].sum(axis=1) == 1).all())},
        {"check": "no final clustering csv outputs", "value": final_output_files, "passes": len(final_output_files) == 0},
    ]
)

display(validation_results)
assert validation_results["passes"].all()
print("Validation passed. No clustering was performed and no final clustering output file was found.")


## Feature group inventory

Before selecting modeling features, it helps to group columns by meaning. This inventory shows what exists in the feature table and gives examples from each group.


In [ ]:
spend_amount_columns = [column for column in feature_table.columns if column.startswith("lifetime_spend_")]
spend_share_columns = [column for column in feature_table.columns if column.startswith("spend_share_")]
missingness_columns = [column for column in feature_table.columns if column.endswith("_was_missing")]
quality_flag_columns = [
    column for column in feature_table.columns
    if "suspicious" in column or "missing_or_invalid" in column or column.endswith("_missing")
]

feature_groups = {
    "row key": ["customer_id"],
    "demographic": ["customer_age", "gender_female", "gender_male", "gender_unknown"],
    "degree": degree_columns,
    "household": ["kids_home", "teens_home", "total_children_home", "has_kids_home", "has_teens_home", "has_children_home"],
    "location": ["latitude", "longitude"],
    "tenure": ["customer_tenure_years"],
    "loyalty_complaints_promotion": ["has_loyalty_card", "loyalty_card_missing", "number_complaints", "promotion_pct_clean", "promotion_pct_suspicious"],
    "spend_amounts": ["total_lifetime_spend", *spend_amount_columns],
    "spend_shares": spend_share_columns,
    "basket_behavior": ["basket_count", "avg_basket_size", "median_basket_size", "max_basket_size", "total_basket_items", "unique_basket_products", "has_sampled_basket"],
    "data_quality_missingness": sorted(set(missingness_columns + quality_flag_columns)),
}

inventory_rows = []
for group, columns in feature_groups.items():
    existing_columns = [column for column in columns if column in feature_table.columns]
    inventory_rows.append(
        {
            "feature_group": group,
            "column_count": len(existing_columns),
            "example_features": ", ".join(existing_columns[:6]),
        }
    )

feature_inventory = pd.DataFrame(inventory_rows)
display(feature_inventory)


## Demographic and household EDA

These features describe who the customers are and basic household structure. They can help with interpretation, but demographic variables should be reviewed carefully before becoming modeling inputs.


In [ ]:
demographic_columns = ["customer_age", "kids_home", "teens_home", "total_children_home"]
demographic_summary = feature_table[demographic_columns].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)

gender_distribution = feature_table[["gender_female", "gender_male", "gender_unknown"]].sum().rename("customer_count").to_frame()
gender_distribution["percent"] = (gender_distribution["customer_count"] / len(feature_table) * 100).round(2)

household_distribution = feature_table[["has_kids_home", "has_teens_home", "has_children_home"]].sum().rename("customer_count").to_frame()
household_distribution["percent"] = (household_distribution["customer_count"] / len(feature_table) * 100).round(2)

print("Demographic and household numeric summary")
display(demographic_summary)
print("Gender indicator distribution")
display(gender_distribution)
print("Household indicator distribution")
display(household_distribution)


## Spend behavior EDA

Spend features are strong candidates for clustering because they describe customer behavior. Raw spend amounts may be skewed, so they should usually be scaled and possibly log-transformed before distance-based clustering.


In [ ]:
spend_summary_columns = ["total_lifetime_spend", *spend_amount_columns]
spend_summary = feature_table[spend_summary_columns].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)

average_spend_shares = feature_table[spend_share_columns].mean().sort_values(ascending=False).rename("mean_share").to_frame()
average_spend_shares["mean_share"] = average_spend_shares["mean_share"].round(3)

print("Spend amount summary")
display(spend_summary)
print("Average spend share by category")
display(average_spend_shares)

if PLOTS_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    feature_table["total_lifetime_spend"].hist(bins=40, ax=axes[0])
    axes[0].set_title("Total lifetime spend")
    axes[0].set_xlabel("Spend")
    axes[0].set_ylabel("Customers")

    average_spend_shares.head(6).sort_values("mean_share").plot(kind="barh", ax=axes[1], legend=False)
    axes[1].set_title("Top average spend shares")
    axes[1].set_xlabel("Mean share")
    plt.tight_layout()
    plt.show()
else:
    print("Optional spend plots skipped because matplotlib is not available in this environment.")


## Basket behavior EDA

Basket features are useful because they describe sampled transaction behavior. The `has_sampled_basket` flag is important because customers without sampled baskets receive zero basket values.


In [ ]:
basket_columns = ["basket_count", "avg_basket_size", "median_basket_size", "max_basket_size", "total_basket_items", "unique_basket_products"]
basket_summary = feature_table[basket_columns].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)

basket_presence = feature_table["has_sampled_basket"].value_counts().rename_axis("has_sampled_basket").reset_index(name="customer_count")
basket_presence["percent"] = (basket_presence["customer_count"] / len(feature_table) * 100).round(2)

print("Basket feature summary")
display(basket_summary)
print("Sampled basket coverage")
display(basket_presence)

if PLOTS_AVAILABLE:
    feature_table["basket_count"].clip(upper=10).hist(bins=10, figsize=(6, 4))
    plt.title("Basket count clipped at 10 for readability")
    plt.xlabel("Sampled basket count")
    plt.ylabel("Customers")
    plt.show()
else:
    print("Optional basket plot skipped because matplotlib is not available in this environment.")


## Loyalty, complaints, and promotion EDA

These features describe engagement and possible dissatisfaction. They can be useful for profiling clusters and may also be modeling candidates if the project wants segments that reflect loyalty and promotion behavior.


In [ ]:
loyalty_columns = ["has_loyalty_card", "loyalty_card_missing", "number_complaints", "promotion_pct_clean", "promotion_pct_suspicious", "promotion_pct_missing"]
loyalty_summary = feature_table[loyalty_columns].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].round(3)

loyalty_distribution = feature_table[["has_loyalty_card", "loyalty_card_missing", "promotion_pct_suspicious", "promotion_pct_missing"]].sum().rename("customer_count").to_frame()
loyalty_distribution["percent"] = (loyalty_distribution["customer_count"] / len(feature_table) * 100).round(2)

print("Loyalty, complaints, and promotion summary")
display(loyalty_summary)
print("Indicator counts")
display(loyalty_distribution)


## Degree feature check

The raw `customer_name` column is not used as a modeling field. Only the academic degree prefix is extracted into four clean numeric indicators. This keeps the useful signal while avoiding direct use of raw names.


In [ ]:
degree_distribution = feature_table[degree_columns].sum().rename("customer_count").to_frame()
degree_distribution["percent"] = (degree_distribution["customer_count"] / len(feature_table) * 100).round(2)

degree_flag_sums = feature_table[degree_columns].sum(axis=1)
print(f"Rows with exactly one degree flag: {(degree_flag_sums == 1).sum():,} of {len(feature_table):,}")
display(degree_distribution)


## Outlier and skewness review

Distance-based clustering is sensitive to scale and skew. This review highlights features that may need scaling, log transformation, clipping, or careful interpretation before modeling.


In [ ]:
def iqr_outlier_share(values):
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return float(((values < lower) | (values > upper)).mean())

outlier_columns = [
    "total_lifetime_spend",
    "lifetime_total_distinct_products",
    "basket_count",
    "total_basket_items",
    "unique_basket_products",
    "number_complaints",
    "promotion_pct_clean",
]

outlier_review = []
for column in outlier_columns:
    values = pd.to_numeric(feature_table[column], errors="coerce")
    outlier_review.append(
        {
            "feature": column,
            "skewness": round(float(values.skew()), 3),
            "iqr_outlier_share": round(iqr_outlier_share(values), 3),
            "max": round(float(values.max()), 3),
        }
    )

skew_review = feature_table.select_dtypes(include="number").skew().abs().sort_values(ascending=False).head(12).rename("abs_skewness").to_frame()
skew_review["abs_skewness"] = skew_review["abs_skewness"].round(3)

print("Selected outlier review")
display(pd.DataFrame(outlier_review))
print("Most skewed numeric features")
display(skew_review)


## Correlation and redundancy review

Correlation helps us see when two features may be repeating the same information. This is important before clustering because distance-based models can give too much weight to duplicated behavior.

Interpretation rules used here:

- 0.70 to 0.85: moderate/high correlation, review carefully.
- Above 0.85: strong redundancy risk, choose one feature or justify keeping both.
- Above 0.95: near-duplicate, normally remove one.

This review does not remove columns automatically. It gives evidence for the manual feature-set decisions below.


In [ ]:
identifier_and_raw_candidates = [
    "customer_id",
    "customer_name",
    "customer_birthdate",
    "customer_birthdate_parsed",
    "loyalty_card_number",
    "year_first_transaction",
    "first_transaction_year_clean",
    "percentage_of_products_bought_promotion",
    "list_of_goods",
]
identifier_and_raw_present = [column for column in identifier_and_raw_candidates if column in feature_table.columns]

candidate_numeric_features = [
    column for column in feature_table.select_dtypes(include="number").columns
    if column not in identifier_and_raw_present
]

correlation_matrix = feature_table[candidate_numeric_features].corr()
upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

correlation_pairs = upper_triangle.stack().reset_index()
correlation_pairs.columns = ["feature_1", "feature_2", "correlation"]
correlation_pairs["abs_correlation"] = correlation_pairs["correlation"].abs()
correlation_pairs = correlation_pairs.sort_values("abs_correlation", ascending=False).reset_index(drop=True)

highly_correlated_pairs = correlation_pairs[correlation_pairs["abs_correlation"] >= 0.70].copy()
highly_correlated_pairs["correlation"] = highly_correlated_pairs["correlation"].round(3)
highly_correlated_pairs["abs_correlation"] = highly_correlated_pairs["abs_correlation"].round(3)

print(f"Candidate numeric features reviewed: {len(candidate_numeric_features)}")
print(f"Feature pairs with absolute correlation >= 0.70: {len(highly_correlated_pairs)}")
display(highly_correlated_pairs.head(30))


## Manual redundancy decision table

The correlation table shows that some groups contain overlapping information. The first baseline should be simple enough to explain, so these decisions deliberately avoid using every available column.

Main decisions:

- `customer_id` and raw identifiers are excluded because they identify rows, not customer behavior.
- Spend is represented as `total_lifetime_spend` plus spend-share features. This keeps one magnitude feature and category preference features without also adding every raw spend column.
- Basket features are tested in a separate feature set because basket data is sampled and some basket measures are strongly correlated.
- Degree indicators are kept for profiling and sensitivity first, rather than forced into the first baseline.
- Gender, location, `typical_hour`, and data-quality flags are profiling-only for now.
- Quality and missingness flags can create artificial clusters if the model groups customers by data problems rather than real behavior.


In [ ]:
redundancy_decisions = pd.DataFrame(
    [
        {
            "feature_group": "identifiers",
            "keep_for_modeling": "none",
            "profiling_only_or_exclude": "customer_id excluded from modeling and kept only as row key",
            "reason": "Identifiers do not measure customer behavior and can distort clustering.",
        },
        {
            "feature_group": "raw identifier fields",
            "keep_for_modeling": "none",
            "profiling_only_or_exclude": "customer_name, birthdate, loyalty number, raw transaction year excluded if present",
            "reason": "Raw fields are either identifiers or unclean source representations already converted into engineered features.",
        },
        {
            "feature_group": "spend magnitude",
            "keep_for_modeling": "total_lifetime_spend",
            "profiling_only_or_exclude": "individual raw lifetime_spend_* amounts reviewed carefully",
            "reason": "One total spend feature captures customer value without over-weighting every product category amount.",
        },
        {
            "feature_group": "spend shares",
            "keep_for_modeling": "all spend_share_* columns",
            "profiling_only_or_exclude": "none for baseline review",
            "reason": "Shares describe category preference and complement total spend.",
        },
        {
            "feature_group": "basket behavior",
            "keep_for_modeling": "has_sampled_basket, basket_count, avg_basket_size, unique_basket_products in feature set B only",
            "profiling_only_or_exclude": "median_basket_size, max_basket_size, total_basket_items initially profiling/excluded",
            "reason": "Basket features are useful but sampled; several are strongly correlated, so the baseline keeps a reduced set.",
        },
        {
            "feature_group": "household features",
            "keep_for_modeling": "kids_home, teens_home",
            "profiling_only_or_exclude": "total_children_home, has_kids_home, has_teens_home, has_children_home",
            "reason": "Kids and teens are simple counts; derived household flags are mostly interpretation helpers.",
        },
        {
            "feature_group": "age and tenure",
            "keep_for_modeling": "customer_age, customer_tenure_years",
            "profiling_only_or_exclude": "age and tenure quality flags",
            "reason": "Age and tenure are interpretable lifecycle features; quality flags should not drive the first baseline.",
        },
        {
            "feature_group": "loyalty, complaints, and promotion",
            "keep_for_modeling": "has_loyalty_card, number_complaints, promotion_pct_clean",
            "profiling_only_or_exclude": "loyalty_card_missing, promotion_pct_missing, promotion_pct_suspicious",
            "reason": "The clean behavior features are useful; missingness and suspicious-value flags are better used for interpretation first.",
        },
        {
            "feature_group": "degree indicators",
            "keep_for_modeling": "not in first baseline",
            "profiling_only_or_exclude": "degree_bsc, degree_msc, degree_phd, degree_unknown",
            "reason": "Degree is an assignment-specific signal, but it should first be used to profile and test sensitivity rather than force the baseline.",
        },
        {
            "feature_group": "gender indicators",
            "keep_for_modeling": "not in first baseline",
            "profiling_only_or_exclude": "gender_female, gender_male, gender_unknown",
            "reason": "Gender is useful for interpretation but should not drive the initial behavior-focused segmentation.",
        },
        {
            "feature_group": "location",
            "keep_for_modeling": "not in first baseline",
            "profiling_only_or_exclude": "latitude, longitude",
            "reason": "Raw coordinates need a clearer location strategy before distance-based clustering.",
        },
        {
            "feature_group": "typical_hour",
            "keep_for_modeling": "not in first baseline",
            "profiling_only_or_exclude": "typical_hour",
            "reason": "Hour-of-day may be useful later, but it is cyclic and should not be treated as a normal numeric scale without preparation.",
        },
        {
            "feature_group": "data quality and missingness flags",
            "keep_for_modeling": "not in first baseline",
            "profiling_only_or_exclude": "missingness, suspicious-value, and parse-failure flags",
            "reason": "These flags can create artificial clusters based on data quality rather than real customer behavior.",
        },
    ]
)

display(redundancy_decisions)


## Baseline modeling feature sets

The next clustering notebook should start with explicit feature lists instead of selecting columns automatically. Two baseline sets are defined here:

- `model_features_a_no_basket`: a clean first baseline from customer-level behavior and demographics, without basket features.
- `model_features_b_with_basket`: the same baseline plus a reduced basket feature set.

Basket features are separated because the basket table is sampled. This lets the project compare whether basket behavior improves the segmentation without mixing that decision into the first baseline.


In [ ]:
def existing(columns):
    # Keep only columns that exist in the current feature table.
    return [column for column in columns if column in feature_table.columns]

all_basket_feature_columns = [
    "has_sampled_basket",
    "basket_count",
    "avg_basket_size",
    "median_basket_size",
    "max_basket_size",
    "total_basket_items",
    "unique_basket_products",
]

base_no_basket_candidates = [
    "customer_age",
    "customer_tenure_years",
    "kids_home",
    "teens_home",
    "distinct_stores_visited",
    "lifetime_total_distinct_products",
    "has_loyalty_card",
    "number_complaints",
    "promotion_pct_clean",
    "total_lifetime_spend",
]

reduced_basket_candidates = [
    "has_sampled_basket",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

profiling_candidate_columns = [
    "gender_female",
    "gender_male",
    "gender_unknown",
    *degree_columns,
    "latitude",
    "longitude",
    "typical_hour",
    "total_children_home",
    "has_children_home",
    "has_kids_home",
    "has_teens_home",
    "loyalty_card_missing",
    "promotion_pct_missing",
    "promotion_pct_suspicious",
    "customer_birthdate_missing",
    "customer_birthdate_parse_failed",
    "customer_age_suspicious",
    "customer_age_missing_or_invalid",
    "first_transaction_year_suspicious",
    "customer_tenure_missing_or_invalid",
    *missingness_columns,
    *quality_flag_columns,
]

excluded_feature_candidates = [
    "customer_id",
    "customer_name",
    "customer_birthdate",
    "customer_birthdate_parsed",
    "loyalty_card_number",
    "year_first_transaction",
    "first_transaction_year_clean",
    "percentage_of_products_bought_promotion",
    "list_of_goods",
]

model_features_a_no_basket = existing(base_no_basket_candidates + spend_share_columns)
model_features_b_with_basket = model_features_a_no_basket + existing(reduced_basket_candidates)
profiling_features = existing(sorted(set(profiling_candidate_columns)))
excluded_features = existing(excluded_feature_candidates)

feature_set_summary = pd.DataFrame(
    [
        {"feature_set": "model_features_a_no_basket", "feature_count": len(model_features_a_no_basket), "features": ", ".join(model_features_a_no_basket)},
        {"feature_set": "model_features_b_with_basket", "feature_count": len(model_features_b_with_basket), "features": ", ".join(model_features_b_with_basket)},
        {"feature_set": "profiling_features", "feature_count": len(profiling_features), "features": ", ".join(profiling_features)},
        {"feature_set": "excluded_features_present", "feature_count": len(excluded_features), "features": ", ".join(excluded_features)},
    ]
)

display(feature_set_summary)


## Transformation notes for the future clustering notebook

These notes are recommendations only. Full modeling preprocessing is not implemented in this notebook.

The next notebook should probably:

- Apply `log1p` to `total_lifetime_spend` because spend is skewed.
- Consider `log1p` for `basket_count` and `unique_basket_products` in the basket feature set.
- Standardize numeric features before distance-based clustering.
- Keep `customer_id` separate from the modeling matrix.
- Compare the no-basket and reduced-basket baselines before choosing a final feature set.


In [ ]:
transformation_notes = pd.DataFrame(
    [
        {"feature_or_group": "total_lifetime_spend", "suggested_preparation": "log1p then standardize", "why": "Spend is right-skewed and can dominate distances."},
        {"feature_or_group": "basket_count", "suggested_preparation": "consider log1p then standardize", "why": "Basket counts are skewed and sampled."},
        {"feature_or_group": "unique_basket_products", "suggested_preparation": "consider log1p then standardize", "why": "Product variety can be large for heavy basket customers."},
        {"feature_or_group": "spend_share_*", "suggested_preparation": "standardize", "why": "Shares are already bounded but still need comparable scale."},
        {"feature_or_group": "customer_id", "suggested_preparation": "keep outside modeling matrix", "why": "It is a row key, not a customer behavior feature."},
    ]
)

display(transformation_notes)


## Baseline feature-set validation

These checks confirm that the feature lists are valid and that the notebook still stops before clustering.


In [ ]:
modeling_features = set(model_features_a_no_basket + model_features_b_with_basket)
reduced_basket_features_present = existing(reduced_basket_candidates)

outputs_dir = PROJECT_ROOT / "outputs"
final_output_files = []
if outputs_dir.exists():
    for path in outputs_dir.iterdir():
        name = path.name.lower()
        looks_like_final_cluster_output = path.suffix.lower() == ".csv" and (
            "cluster" in name or "segment" in name
        )
        if path.is_file() and looks_like_final_cluster_output:
            final_output_files.append(path.name)

feature_set_validation = pd.DataFrame(
    [
        {"check": "feature table shape remains 33038 x 77", "value": str(feature_table.shape), "passes": feature_table.shape == (33038, 77)},
        {"check": "customer_id is unique", "value": feature_table["customer_id"].is_unique, "passes": feature_table["customer_id"].is_unique},
        {"check": "feature table has no missing values", "value": int(feature_table.isna().sum().sum()), "passes": int(feature_table.isna().sum().sum()) == 0},
        {"check": "all model_features_a_no_basket exist", "value": len(model_features_a_no_basket), "passes": all(column in feature_table.columns for column in model_features_a_no_basket)},
        {"check": "all model_features_b_with_basket exist", "value": len(model_features_b_with_basket), "passes": all(column in feature_table.columns for column in model_features_b_with_basket)},
        {"check": "profiling_features all exist", "value": len(profiling_features), "passes": all(column in feature_table.columns for column in profiling_features)},
        {"check": "excluded features are not used for modeling", "value": sorted(set(excluded_features) & modeling_features), "passes": set(excluded_features).isdisjoint(modeling_features)},
        {"check": "no basket features in no-basket baseline", "value": sorted(set(model_features_a_no_basket) & set(all_basket_feature_columns)), "passes": set(model_features_a_no_basket).isdisjoint(all_basket_feature_columns)},
        {"check": "with-basket baseline contains reduced basket set", "value": reduced_basket_features_present, "passes": set(reduced_basket_features_present).issubset(model_features_b_with_basket)},
        {"check": "customer_id excluded from no-basket baseline", "value": "customer_id" in model_features_a_no_basket, "passes": "customer_id" not in model_features_a_no_basket},
        {"check": "customer_id excluded from with-basket baseline", "value": "customer_id" in model_features_b_with_basket, "passes": "customer_id" not in model_features_b_with_basket},
        {"check": "no final clustering csv outputs", "value": final_output_files, "passes": len(final_output_files) == 0},
    ]
)

display(feature_set_validation)
assert feature_set_validation["passes"].all()
print("Baseline feature-set validation passed. No clustering model was trained and no final cluster CSV was created.")


## Summary and next recommended task

The feature review is now concrete enough to support the first clustering experiment design. The project has two explicit baseline feature sets and a clear list of profiling and excluded features.

The next task should create a clustering preparation notebook that applies transformations, scales the selected features, and compares the no-basket baseline against the reduced-basket baseline. That future notebook can train the first clustering model, but this notebook does not.


In [ ]:
summary = pd.DataFrame(
    [
        {"topic": "feature table", "finding": "33,038 customers and 77 columns after preprocessing and feature engineering."},
        {"topic": "correlation review", "finding": "Highly correlated pairs were reviewed so redundant features can be handled before clustering."},
        {"topic": "baseline A", "finding": f"model_features_a_no_basket contains {len(model_features_a_no_basket)} customer_info-derived features and no basket features."},
        {"topic": "baseline B", "finding": f"model_features_b_with_basket contains {len(model_features_b_with_basket)} features, including the reduced basket set."},
        {"topic": "profiling", "finding": "Gender, degree, location, typical hour, household flags, and data-quality flags are kept for interpretation/sensitivity first."},
        {"topic": "next task", "finding": "Create the clustering preparation notebook: transform, scale, then train a first baseline model."},
    ]
)

display(summary)
print("EDA feature review refinement complete. No clustering model was trained and no final cluster CSV was created.")
